# Diabetic Retinopathy Grading with a Dual-Branch Swin Transformer

This notebook contains the complete training and evaluation pipeline used for the research
experiments. A base Swin Transformer first learns from retinal RGB images. Integrated Gradients
then produces lesion-attribution maps, which are paired with the RGB images to train the final
dual-branch classifier.

The executable workflow includes:

1. deterministic preprocessing and paired image/heatmap augmentation;
2. ordinal-smoothed, class-weighted focal loss;
3. base-model training and Integrated Gradients heatmap generation;
4. dual-branch training, checkpointing, and test-time augmentation;
5. ablation experiments, stratified cross-validation, statistical summaries, and export.

> **Execution requirements:** Use a CUDA-capable environment for practical runtimes and place
> the APTOS split files under `aptos/` as documented in the driver cell. Run cells from top to
> bottom. The ablation and cross-validation sections retrain multiple models and can take many hours.

The earlier loopback fine-tuning stage is intentionally absent: the reported ablation showed that
its small gain did not justify a second attribution and fine-tuning pass. The Stage 4 dual-branch
checkpoint is therefore the final model.

## Environment setup

Install a CUDA-compatible PyTorch and torchvision build for your system first. The cell below
installs the remaining notebook dependencies into the active Jupyter kernel.

In [ ]:
%pip install -q timm captum thop scikit-learn seaborn openpyxl tqdm pandas numpy pillow matplotlib

In [ ]:
# =========================================================
# IMPORTS + CONFIG
# =========================================================
import json
import math
import os
import random
import shutil
import timm
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from thop import profile
from captum.attr import IntegratedGradients
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

NUM_CLASSES = 5
BATCH_SIZE  = 16
EPOCHS_BASE = 15
EPOCHS_DUAL = 12
# EPOCHS_LOOP removed -- loopback fine-tuning stage was cut from the
# architecture (ablation showed it added ~0.19pp kappa, smaller than
# the full_model's own seed-to-seed noise of ~0.29pp).

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


In [ ]:
# =========================================================
# ORDINAL LABEL SMOOTHING + CLASS-WEIGHTED FOCAL LOSS
# =========================================================
def make_ordinal_targets(labels, num_classes=NUM_CLASSES,
                          main_w=0.76, adj1_w=0.07, adj2_w=0.015):
    """
    Spreads probability mass onto adjacent DR grades instead of a hard one-hot
    label, matching the ordinal/continuous nature of DR severity, e.g. grade 2
    -> [0.015, 0.07, 0.76, 0.07, 0.015]. Boundary grades (0, 4) have fewer
    neighbours, so each row is renormalized to sum to 1.
    """
    targets = torch.zeros(len(labels), num_classes, device=labels.device)

    for i in range(len(labels)):
        l = labels[i].item()
        for c in range(num_classes):
            dist = abs(c - l)
            if dist == 0:
                targets[i, c] = main_w
            elif dist == 1:
                targets[i, c] = adj1_w
            elif dist == 2:
                targets[i, c] = adj2_w

    targets = targets / targets.sum(dim=1, keepdim=True)
    return targets


class FocalLoss(nn.Module):
    """
    Focal loss against ordinal-smoothed soft targets, with a per-sample
    multiply by the true grade's class weight:
        loss = (1 - exp(-CE))^gamma * CE * class_weight[true_grade]
    """
    def __init__(self, gamma=2.0, weight=None, num_classes=NUM_CLASSES,
                 use_ordinal_smoothing=True):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.num_classes = num_classes
        self.use_ordinal_smoothing = use_ordinal_smoothing

    def forward(self, logits, targets):
        if self.use_ordinal_smoothing:
            soft_targets = make_ordinal_targets(targets, self.num_classes)
        else:
            soft_targets = F.one_hot(targets, self.num_classes).float()

        log_probs = F.log_softmax(logits, dim=1)
        ce = -(soft_targets * log_probs).sum(dim=1)

        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce

        if self.weight is not None:
            loss = loss * self.weight[targets]

        return loss.mean()


In [ ]:
# =========================================================
# DATASET + AUGMENTATION
#
# Geometric augmentation (crop/flip/rotate) is applied identically to the
# image AND the lesion heatmap so branch1/branch2 stay spatially aligned.
# Color jitter is photometric and only applied to the image.
# =========================================================
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

_color_jitter = transforms.ColorJitter(
    brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05
)


def paired_train_augment(img_pil, heatmap_np):
    img = TF.resize(img_pil, (256, 256))

    heatmap_img = None
    if heatmap_np is not None:
        heatmap_img = Image.fromarray((heatmap_np * 255).astype(np.uint8))
        heatmap_img = TF.resize(heatmap_img, (256, 256))

    i, j, h, w = transforms.RandomCrop.get_params(img, output_size=(224, 224))
    img = TF.crop(img, i, j, h, w)
    if heatmap_img is not None:
        heatmap_img = TF.crop(heatmap_img, i, j, h, w)

    if random.random() < 0.5:
        img = TF.hflip(img)
        if heatmap_img is not None:
            heatmap_img = TF.hflip(heatmap_img)

    if random.random() < 0.5:
        img = TF.vflip(img)
        if heatmap_img is not None:
            heatmap_img = TF.vflip(heatmap_img)

    angle = random.uniform(0, 360)
    img = TF.rotate(img, angle)
    if heatmap_img is not None:
        heatmap_img = TF.rotate(heatmap_img, angle)

    img = _color_jitter(img)  # photometric only, image branch

    img = TF.to_tensor(img)
    img = TF.normalize(img, IMAGENET_MEAN, IMAGENET_STD)

    if heatmap_img is not None:
        heatmap_t = TF.to_tensor(heatmap_img).squeeze(0)  # back to [0,1], HxW
        lesion = heatmap_t.unsqueeze(0).repeat(3, 1, 1)
    else:
        lesion = torch.zeros(3, 224, 224)

    return img, lesion


class EyeDataset(Dataset):
    def __init__(self, df, image_dir, heatmap_dir=None, augment=False):
        self.image_paths = [os.path.join(image_dir, f) for f in df["id_code"]]
        self.labels = df["diagnosis"].values
        self.heatmap_dir = heatmap_dir
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def _load_heatmap(self, idx):
        if not self.heatmap_dir:
            return None
        fname = os.path.splitext(os.path.basename(self.image_paths[idx]))[0] + ".npy"
        path = os.path.join(self.heatmap_dir, fname)
        if os.path.exists(path):
            return np.load(path)
        return None  # falls back to zero lesion tensor; only expected if a heatmap is missing

    def __getitem__(self, idx):
        img_pil = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        heatmap_np = self._load_heatmap(idx)

        if self.augment:
            img, lesion = paired_train_augment(img_pil, heatmap_np)
        else:
            img = eval_transform(img_pil)
            if heatmap_np is not None:
                lesion = torch.tensor(heatmap_np, dtype=torch.float32)
                lesion = lesion.unsqueeze(0).repeat(3, 1, 1)
                if lesion.shape[-2:] != (224, 224):
                    lesion = F.interpolate(lesion.unsqueeze(0), size=(224, 224),
                                            mode="bilinear", align_corners=False).squeeze(0)
            else:
                lesion = torch.zeros(3, 224, 224)

        return img, torch.tensor(label), lesion


In [ ]:
# =========================================================
# SWIN BACKBONE + MODELS
# =========================================================
class SwinBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

    def forward(self, x):
        return self.model(x)


class BaseSwin(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = SwinBackbone()
        self.norm = nn.LayerNorm(768)
        self.fc = nn.Linear(768, num_classes)

    def forward(self, x):
        feat = self.backbone(x)
        feat = self.norm(feat)
        return self.fc(feat)


class DualBranchSwin(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.branch1 = SwinBackbone()  # RGB, warm-started from best_base.pth
        self.branch2 = SwinBackbone()  # lesion heatmap, randomly initialized

        self.classifier = nn.Sequential(
            nn.LayerNorm(1536),
            nn.Linear(1536, 512),
            nn.GELU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes),
        )

    def forward(self, img, lesion):
        f1 = self.branch1(img)
        f2 = self.branch2(lesion)
        fused = torch.cat([f1, f2], dim=1)
        return self.classifier(fused)


In [ ]:
# =========================================================
# INTEGRATED GRADIENTS HEATMAPS
# =========================================================
def compute_ig_heatmap(model, image, target, n_steps=4):
    if isinstance(model, DualBranchSwin):
        ig = IntegratedGradients(lambda x: model(x, x))
    else:
        ig = IntegratedGradients(model)

    blur = TF.gaussian_blur(image.squeeze(0), 7)
    baseline = blur.unsqueeze(0).to(DEVICE)

    attributions = ig.attribute(
        image, baselines=baseline, target=int(target), n_steps=n_steps
    )

    heatmap = attributions.sum(dim=1, keepdim=True)
    heatmap = F.relu(heatmap)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    return heatmap.squeeze().detach().cpu().numpy()


def generate_heatmaps(model, dataset, save_dir):
    """`dataset` must use augment=False (deterministic) so heatmaps stay
    aligned to the raw image grid regardless of which split it's called on."""
    os.makedirs(save_dir, exist_ok=True)
    model.eval()

    for i in tqdm(range(len(dataset))):
        img, label, _ = dataset[i]
        img = img.unsqueeze(0).to(DEVICE)

        heatmap = compute_ig_heatmap(model, img, label)

        fname = os.path.splitext(os.path.basename(dataset.image_paths[i]))[0] + ".npy"
        np.save(os.path.join(save_dir, fname), heatmap)


In [ ]:
# =========================================================
# TRAIN / VALIDATE / STAGE RUNNER
# =========================================================
def train_epoch(model, loader, optimizer, class_weights, dual=False, grad_clip=1.0):
    model.train()
    criterion = FocalLoss(weight=class_weights)

    total_loss, correct, total = 0.0, 0, 0

    for img, label, lesion in loader:
        img, label = img.to(DEVICE), label.to(DEVICE)

        optimizer.zero_grad()

        if dual:
            lesion = lesion.to(DEVICE)
            out = model(img, lesion)
        else:
            out = model(img)

        loss = criterion(out, label)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item()
        pred = out.argmax(1)
        correct += (pred == label).sum().item()
        total += label.size(0)

    return total_loss / len(loader), correct / total


def validate(model, loader, class_weights, dual=False):
    model.eval()
    criterion = FocalLoss(weight=class_weights)

    total_loss, correct, total = 0.0, 0, 0
    y_true, y_pred = [], []

    with torch.no_grad():
        for img, label, lesion in loader:
            img, label = img.to(DEVICE), label.to(DEVICE)

            if dual:
                lesion = lesion.to(DEVICE)
                out = model(img, lesion)
            else:
                out = model(img)

            loss = criterion(out, label)
            total_loss += loss.item()

            pred = out.argmax(1)
            correct += (pred == label).sum().item()
            total += label.size(0)

            y_true.extend(label.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    acc = correct / total
    loss = total_loss / len(loader)
    kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    return loss, acc, kappa


def run_stage(model, train_loader, val_loader, optimizer, scheduler, class_weights,
              dual, epochs, patience, ckpt_path, history=None):
    """Shared training loop: trains up to `epochs`, early-stops on a stalled
    validation kappa, checkpoints the best-kappa weights to ckpt_path, and
    reloads them before returning."""
    best_kappa = -1.0
    counter = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, class_weights, dual=dual)
        val_loss, val_acc, val_kappa = validate(model, val_loader, class_weights, dual=dual)

        if scheduler is not None:
            scheduler.step()

        print(f"Epoch {epoch}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}  val_kappa={val_kappa:.4f}")

        if history is not None:
            history["train_loss"].append(train_loss)
            history["train_acc"].append(train_acc)
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            history["val_kappa"].append(val_kappa)

        if val_kappa > best_kappa:
            best_kappa = val_kappa
            counter = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            counter += 1
            if counter >= patience:
                print("Early stopping triggered")
                break

    model.load_state_dict(torch.load(ckpt_path))
    return model, best_kappa


def plot_training(history):
    epochs = range(len(history["train_loss"]))
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.plot(epochs, history["train_loss"], label="Train")
    plt.plot(epochs, history["val_loss"], label="Validation")
    plt.title("Loss"); plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, history["train_acc"], label="Train")
    plt.plot(epochs, history["val_acc"], label="Validation")
    plt.title("Accuracy"); plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, history["val_kappa"], label="Validation")
    plt.title("Quadratic Weighted Kappa"); plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# =========================================================
# TEST-TIME AUGMENTATION (8-pass, h/v flip averaging)
# =========================================================
def tta_predict_batch(model, img, lesion, dual, n_passes=8):
    probs_sum = 0
    flip_combos = [(False, False), (True, False), (False, True), (True, True)]

    with torch.no_grad():
        for p in range(n_passes):
            do_h, do_v = flip_combos[p % 4]

            aug_img, aug_lesion = img, lesion
            if do_h:
                aug_img = TF.hflip(aug_img)
                aug_lesion = TF.hflip(aug_lesion)
            if do_v:
                aug_img = TF.vflip(aug_img)
                aug_lesion = TF.vflip(aug_lesion)

            out = model(aug_img, aug_lesion) if dual else model(aug_img)
            probs_sum = probs_sum + F.softmax(out, dim=1)

    return probs_sum / n_passes


def evaluate_tta(model, loader, dual=True, n_passes=8, return_predictions=False):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for img, label, lesion in loader:
            img = img.to(DEVICE)
            lesion = lesion.to(DEVICE)

            probs = tta_predict_batch(model, img, lesion, dual=dual, n_passes=n_passes)
            pred = probs.argmax(1)

            y_true.extend(label.numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true, y_pred = np.array(y_true), np.array(y_pred)

    acc = (y_true == y_pred).mean()
    kappa = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    print("\nFinal Accuracy (TTA):", acc)
    print("Quadratic Weighted Kappa (TTA):", kappa)
    print("\nClassification Report")
    print(classification_report(y_true, y_pred))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (TTA)")
    plt.show()

    if return_predictions:
        return acc, kappa, y_true.tolist(), y_pred.tolist()
    return acc, kappa


def evaluate_tta_and_save(model, loader, run_name, dual=True, n_passes=8):
    """Evaluate once and persist per-sample labels for later statistical tests."""
    acc, kappa, y_true, y_pred = evaluate_tta(
        model, loader, dual=dual, n_passes=n_passes, return_predictions=True
    )

    out_path = os.path.join(RESULTS_DIR, f"{run_name}_preds.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "run_name": run_name,
            "acc": float(acc),
            "kappa": float(kappa),
            "y_true": y_true,
            "y_pred": y_pred,
        }, f)

    print(f"Saved predictions -> {out_path}")
    return acc, kappa, y_true, y_pred


## Model complexity

The final helper reports parameter count and approximate FLOPs for either the base or dual-branch model.

In [ ]:
# =========================================================
# MODEL COMPLEXITY
# =========================================================
def model_stats(model, input_size=(1, 3, 224, 224), dual=False):
    model.eval()
    dummy = torch.randn(input_size).to(DEVICE)

    if dual:
        flops, params = profile(model, inputs=(dummy, dummy), verbose=False)
    else:
        flops, params = profile(model, inputs=(dummy,), verbose=False)

    print(f"Params: {params/1e6:.2f}M")
    print(f"FLOPs: {flops/1e9:.2f}G")
    return flops, params


## Driver

In [ ]:
# =========================================================
# MAIN DRIVER
# =========================================================
def main():
    train_csv = "./aptos/train.csv"
    val_csv   = "./aptos/valid.csv"
    test_csv  = "./aptos/test.csv"

    train_dir = "./aptos/train_images"
    val_dir   = "./aptos/val_images"
    test_dir  = "./aptos/test_images"

    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv)

    train_df["id_code"] = train_df["id_code"] + ".png"
    val_df["id_code"]   = val_df["id_code"] + ".png"
    test_df["id_code"]  = test_df["id_code"] + ".png"

    # ================= CLASS WEIGHTS / SAMPLER =================
    class_counts = np.bincount(train_df["diagnosis"], minlength=NUM_CLASSES)
    class_weights = 1.0 / np.sqrt(class_counts)
    class_weights = class_weights / class_weights.sum() * NUM_CLASSES
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    sample_weights = class_weights.cpu()[train_df["diagnosis"].values]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    # =====================================================
    # STAGE 2 — BASE SWIN TRAINING
    # =====================================================
    train_dataset_aug = EyeDataset(train_df, train_dir, augment=True)
    train_loader = DataLoader(train_dataset_aug, batch_size=BATCH_SIZE,
                               sampler=sampler, num_workers=4, pin_memory=True)

    val_dataset = EyeDataset(val_df, val_dir, augment=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    base_model = BaseSwin(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(base_model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )

    print("\n=== Stage 2: Training Base Swin ===")
    base_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_kappa": []}
    base_model, base_kappa = run_stage(
        base_model, train_loader, val_loader, optimizer, scheduler, class_weights,
        dual=False, epochs=EPOCHS_BASE, patience=5, ckpt_path="best_base.pth", history=base_history
    )
    print(f"Best base val kappa: {base_kappa:.4f}")

    # =====================================================
    # STAGE 3 — HEATMAP V1 (train + val + test, deterministic pass)
    # =====================================================
    print("\n=== Stage 3: Generating heatmaps v1 ===")
    generate_heatmaps(base_model, EyeDataset(train_df, train_dir, augment=False), "heatmaps_v1")
    generate_heatmaps(base_model, EyeDataset(val_df,   val_dir,   augment=False), "heatmaps_v1")
    generate_heatmaps(base_model, EyeDataset(test_df,  test_dir,  augment=False), "heatmaps_v1")

    # =====================================================
    # STAGE 4 — DUAL-BRANCH TRAINING
    # =====================================================
    dual_train_aug = EyeDataset(train_df, train_dir, heatmap_dir="heatmaps_v1", augment=True)
    dual_loader = DataLoader(dual_train_aug, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=4, pin_memory=True)

    dual_val_dataset = EyeDataset(val_df, val_dir, heatmap_dir="heatmaps_v1", augment=False)
    dual_val_loader = DataLoader(dual_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    dual_model = DualBranchSwin(NUM_CLASSES).to(DEVICE)
    dual_model.branch1.load_state_dict(base_model.backbone.state_dict())

    optimizer2 = torch.optim.AdamW(dual_model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler2 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer2, T_0=6, T_mult=2, eta_min=1e-7
    )

    print("\n=== Stage 4: Training Dual Swin ===")
    dual_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_kappa": []}
    dual_model, dual_kappa = run_stage(
        dual_model, dual_loader, dual_val_loader, optimizer2, scheduler2, class_weights,
        dual=True, epochs=EPOCHS_DUAL, patience=5, ckpt_path="best_dual.pth", history=dual_history
    )
    print(f"Best dual val kappa: {dual_kappa:.4f}")

    # dual-branch model is now the FINAL model -- loopback fine-tuning was
    # cut (ablation: ~0.19pp kappa gain, smaller than the run-to-run noise).
    shutil.copy("best_dual.pth", "best_model.pth")

    # =====================================================
    # STAGE 5 — TTA EVALUATION ON TEST SET
    # (loopback fine-tuning removed; dual_model / best_model.pth is final)
    # =====================================================
    test_dataset = EyeDataset(test_df, test_dir, heatmap_dir="heatmaps_v1", augment=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print("\n=== FINAL TEST RESULTS (8-pass TTA) ===")
    test_acc, test_kappa = evaluate_tta(dual_model, test_loader, dual=True, n_passes=8)

    print("\nModel Complexity")
    model_stats(dual_model, dual=True)

    plot_training(dual_history)

    return {
        "base_kappa": base_kappa,
        "dual_kappa": dual_kappa,
        "test_acc": test_acc,
        "test_kappa": test_kappa,
    }


if __name__ == "__main__":
    results = main()

## Statistical utilities

The evaluation helper above saves exact per-sample labels and predictions for TTA-enabled
experiment runs. The functions below compute Wilson confidence intervals directly from those
saved predictions instead of relying on manually transcribed classification-report values.
Call `summarize_saved_predictions(...)` after the corresponding experiment has completed.

In [ ]:
# =========================================================
# WILSON CONFIDENCE INTERVALS FROM SAVED PREDICTIONS
# =========================================================
def wilson_ci(correct, n, z=1.96):
    """95% Wilson score interval for a binomial proportion."""
    phat = correct / n
    denom = 1 + z**2 / n
    center = phat + z**2 / (2 * n)
    half = z * math.sqrt(phat * (1 - phat) / n + z**2 / (4 * n**2))
    return phat, (center - half) / denom, (center + half) / denom


def summarize_saved_predictions(prediction_path):
    """Print overall and per-grade Wilson intervals from a saved prediction JSON."""
    with open(prediction_path, encoding="utf-8") as f:
        payload = json.load(f)

    y_true = np.asarray(payload["y_true"])
    y_pred = np.asarray(payload["y_pred"])
    rows = []

    correct = int((y_true == y_pred).sum())
    p, lo, hi = wilson_ci(correct, len(y_true))
    rows.append({"group": "overall", "estimate": p, "ci_low": lo, "ci_high": hi, "n": len(y_true)})

    for grade in range(NUM_CLASSES):
        mask = y_true == grade
        support = int(mask.sum())
        if support == 0:
            continue
        grade_correct = int((y_pred[mask] == grade).sum())
        p, lo, hi = wilson_ci(grade_correct, support)
        rows.append({"group": f"grade_{grade}_recall", "estimate": p, "ci_low": lo, "ci_high": hi, "n": support})

    summary = pd.DataFrame(rows)
    print(summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    return summary


## Ablation study

The sweep evaluates the full configuration plus six single-component ablations using the same
seed and reduced base/dual epoch budgets. This controls the comparison while keeping the sweep
computationally feasible. These reduced-budget results must be reported separately from the main
15/12-epoch run.

The published configuration uses one seed per ablation (`ABLATION_SEED = 0`). Treat those values
as sensitivity evidence rather than a variance estimate; use multiple seeds before making strong
claims about small differences.

In [ ]:
# =========================================================
# ABLATION CONFIGS
# =========================================================
ABLATION_EPOCH_BUDGET = dict(base=6, dual=5)  # shared, reduced budget for the sweep (loop removed)
ABLATION_SEED = 0  # single seed per config -- no seed sweep

# use_loopback removed -- loopback fine-tuning was cut from the
# architecture entirely (ablation: ~0.19pp kappa gain, smaller than
# full_model's own seed-to-seed noise of ~0.29pp). The dual-branch
# stage is now the final stage for every config below.
FULL_CONFIG = dict(
    use_augmentation=True,
    use_ordinal_smoothing=True,
    use_dual_branch=True,
    use_tta=True,
    use_class_weighting=True,
    use_cosine_scheduler=True,
)

ABLATION_CONFIGS = {
    "full_model":            dict(FULL_CONFIG),
    "no_augmentation":       {**FULL_CONFIG, "use_augmentation": False},
    "no_ordinal_smoothing":  {**FULL_CONFIG, "use_ordinal_smoothing": False},
    "no_dual_branch":        {**FULL_CONFIG, "use_dual_branch": False},
    "no_tta":                {**FULL_CONFIG, "use_tta": False},
    "no_class_weighting":    {**FULL_CONFIG, "use_class_weighting": False},
    "no_cosine_scheduler":   {**FULL_CONFIG, "use_cosine_scheduler": False},
}

for name, cfg in ABLATION_CONFIGS.items():
    print(name, "->", {k: v for k, v in cfg.items() if v != FULL_CONFIG[k]} or "(baseline)")


In [ ]:
# =========================================================
# ABLATION RUNNER
#
# Reuses train_epoch / validate / run_stage / EyeDataset / BaseSwin /
# DualBranchSwin / FocalLoss / evaluate_tta_and_save from the cells
# above -- this just threads the config flags through them.
# =========================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_ablation_config(name, cfg, seed, train_df, val_df, test_df,
                         train_dir, val_dir, test_dir, epoch_budget=ABLATION_EPOCH_BUDGET):
    set_seed(seed)
    run_name = f"{name}_seed{seed}"

    class_counts = np.bincount(train_df["diagnosis"], minlength=NUM_CLASSES)
    if cfg["use_class_weighting"]:
        class_weights = 1.0 / np.sqrt(class_counts)
        class_weights = class_weights / class_weights.sum() * NUM_CLASSES
        class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
        sample_weights = class_weights.cpu()[train_df["diagnosis"].values]
        sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
        loader_kwargs = dict(sampler=sampler)
    else:
        class_weights = None
        loader_kwargs = dict(shuffle=True)

    # augmentation toggle: EyeDataset(augment=...) already does the right
    # thing -- augment=False falls back to eval_transform (deterministic).
    train_dataset = EyeDataset(train_df, train_dir, augment=cfg["use_augmentation"])
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=4,
                               pin_memory=True, **loader_kwargs)
    val_dataset = EyeDataset(val_df, val_dir, augment=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    def make_optimizer_scheduler(params, lr, epochs):
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        if cfg["use_cosine_scheduler"]:
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=lr * 1e-2)
        else:
            sched = None
        return opt, sched

    # temporarily monkeypatch FocalLoss ordinal-smoothing flag for this run
    _orig_forward = FocalLoss.forward
    def _patched_forward(self, logits, targets):
        self.use_ordinal_smoothing = cfg["use_ordinal_smoothing"]
        return _orig_forward(self, logits, targets)
    FocalLoss.forward = _patched_forward

    try:
        # ---- base stage ----
        base_model = BaseSwin(NUM_CLASSES).to(DEVICE)
        opt, sched = make_optimizer_scheduler(base_model.parameters(), 1e-4, epoch_budget["base"])
        base_model, _ = run_stage(base_model, train_loader, val_loader, opt, sched, class_weights,
                                   dual=False, epochs=epoch_budget["base"], patience=epoch_budget["base"],
                                   ckpt_path=f"results/{run_name}_base.pth")

        if not cfg["use_dual_branch"]:
            # single-branch ablation: evaluate the base model directly, no lesion input
            test_dataset = EyeDataset(test_df, test_dir, augment=False)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
            if cfg["use_tta"]:
                acc, kappa, y_true, y_pred = evaluate_tta_and_save(base_model, test_loader, run_name, dual=False)
            else:
                _, acc, kappa = validate(base_model, test_loader, class_weights, dual=False)
                y_true = y_pred = None  # single-pass eval: skip raw-prediction saving for brevity
            return dict(run_name=run_name, config=name, seed=seed, acc=acc, kappa=kappa,
                        y_true=y_true, y_pred=y_pred)

        # ---- heatmaps v1 (train + val + test) ----
        generate_heatmaps(base_model, EyeDataset(train_df, train_dir, augment=False), f"results/{run_name}_hmv1")
        generate_heatmaps(base_model, EyeDataset(val_df, val_dir, augment=False), f"results/{run_name}_hmv1")
        generate_heatmaps(base_model, EyeDataset(test_df, test_dir, augment=False), f"results/{run_name}_hmv1")

        # ---- dual stage ----
        dual_train = EyeDataset(train_df, train_dir, heatmap_dir=f"results/{run_name}_hmv1", augment=cfg["use_augmentation"])
        dual_loader = DataLoader(dual_train, batch_size=BATCH_SIZE, num_workers=4, pin_memory=True, **loader_kwargs)
        dual_val = EyeDataset(val_df, val_dir, heatmap_dir=f"results/{run_name}_hmv1", augment=False)
        dual_val_loader = DataLoader(dual_val, batch_size=BATCH_SIZE, shuffle=False)

        dual_model = DualBranchSwin(NUM_CLASSES).to(DEVICE)
        dual_model.branch1.load_state_dict(base_model.backbone.state_dict())
        opt2, sched2 = make_optimizer_scheduler(dual_model.parameters(), 1e-5, epoch_budget["dual"])
        dual_model, _ = run_stage(dual_model, dual_loader, dual_val_loader, opt2, sched2, class_weights,
                                   dual=True, epochs=epoch_budget["dual"], patience=epoch_budget["dual"],
                                   ckpt_path=f"results/{run_name}_dual.pth")

        # loopback fine-tuning removed from the architecture -- the dual
        # stage above is now the final stage for every config.
        test_dataset = EyeDataset(test_df, test_dir, heatmap_dir=f"results/{run_name}_hmv1", augment=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        if cfg["use_tta"]:
            acc, kappa, y_true, y_pred = evaluate_tta_and_save(dual_model, test_loader, run_name, dual=True)
        else:
            _, acc, kappa = validate(dual_model, test_loader, class_weights, dual=True)
            y_true = y_pred = None
        return dict(run_name=run_name, config=name, seed=seed, acc=acc, kappa=kappa,
                    y_true=y_true, y_pred=y_pred)

    finally:
        FocalLoss.forward = _orig_forward  # always undo the monkeypatch


In [ ]:
# =========================================================
# RUN THE SWEEP
#
# 7 configs, each run ONCE (seed=0) -- no seed sweep. Still a
# reduced (but still multi-stage) pipeline per config. Consider
# running configs individually / across sessions and appending to
# ablation_results.csv instead of running this cell in one shot.
# =========================================================
train_csv, val_csv, test_csv = "./aptos/train.csv", "./aptos/valid.csv", "./aptos/test.csv"
train_dir, val_dir, test_dir = "./aptos/train_images", "./aptos/val_images", "./aptos/test_images"

train_df = pd.read_csv(train_csv); val_df = pd.read_csv(val_csv); test_df = pd.read_csv(test_csv)
train_df["id_code"] += ".png"; val_df["id_code"] += ".png"; test_df["id_code"] += ".png"

ablation_rows = []
for name, cfg in ABLATION_CONFIGS.items():
    print(f"\n>>> Running {name} seed={ABLATION_SEED}")
    result = run_ablation_config(name, cfg, ABLATION_SEED, train_df, val_df, test_df, train_dir, val_dir, test_dir)
    ablation_rows.append({k: v for k, v in result.items() if k not in ("y_true", "y_pred")})
    pd.DataFrame(ablation_rows).to_csv("results/ablation_results.csv", index=False)  # incremental save

ablation_df = pd.DataFrame(ablation_rows)
# ablation_df


## Cross-validation

Stratified k-fold over `train_df + val_df` combined (the test set stays held out, untouched, for
the final external evaluation each fold reports on). Reuses `run_ablation_config` unchanged --
a fold's train/val split is just another `(train_df, val_df)` pair, so the same pipeline-building
code applies (base -> heatmaps v1 -> dual; no loopback stage, matching the current architecture).
Runs on `full_model` by default; add names to `CV_CONFIGS` for CV checks on ablated variants too.

In [ ]:
# =========================================================
# STRATIFIED K-FOLD CROSS-VALIDATION
# =========================================================
N_FOLDS = 5
CV_SEEDS = [0]                 # add more seeds per fold if you also want seed-variance within CV
CV_CONFIGS = ["full_model"]    # e.g. ["full_model", "no_augmentation"] to CV-check an ablation too


def run_cross_validation(config_name, cfg, n_folds, seeds, train_df, val_df, test_df,
                          train_dir, val_dir, test_dir, epoch_budget=ABLATION_EPOCH_BUDGET,
                          csv_path="results/cv_results.csv"):
    # Resolve to absolute paths up front so train/val images (which live in
    # different directories) can be pooled into one dataframe safely --
    # EyeDataset just does os.path.join(image_dir, id_code), and
    # os.path.join("", "/abs/path.png") returns the absolute path unchanged,
    # so passing image_dir="" downstream works once id_code holds full paths.
    train_abs = train_df.copy()
    train_abs["id_code"] = train_abs["id_code"].apply(lambda f: os.path.abspath(os.path.join(train_dir, f)))
    val_abs = val_df.copy()
    val_abs["id_code"] = val_abs["id_code"].apply(lambda f: os.path.abspath(os.path.join(val_dir, f)))
    cv_pool = pd.concat([train_abs, val_abs], ignore_index=True)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seeds[0])

    rows = []
    if os.path.exists(csv_path):
        rows = pd.read_csv(csv_path).to_dict("records")  # resume support

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(cv_pool, cv_pool["diagnosis"])):
        fold_train_df = cv_pool.iloc[tr_idx].reset_index(drop=True)
        fold_val_df = cv_pool.iloc[va_idx].reset_index(drop=True)

        for seed in seeds:
            run_id = f"{config_name}_cv{fold_idx}"
            already_done = any(r["run_name"] == f"{run_id}_seed{seed}" for r in rows)
            if already_done:
                print(f"Skipping {run_id}_seed{seed} (already in {csv_path})")
                continue

            print(f"\n>>> CV fold {fold_idx}/{n_folds} | config={config_name} | seed={seed}")
            result = run_ablation_config(run_id, cfg, seed, fold_train_df, fold_val_df, test_df,
                                          train_dir="", val_dir="", test_dir=test_dir,
                                          epoch_budget=epoch_budget)

            row = {k: v for k, v in result.items() if k not in ("y_true", "y_pred")}
            row["base_config"] = config_name
            row["fold"] = fold_idx
            rows.append(row)

            pd.DataFrame(rows).to_csv(csv_path, index=False)  # incremental save after every run

    return pd.DataFrame(rows)


cv_rows = []
for config_name in CV_CONFIGS:
    cfg = ABLATION_CONFIGS[config_name]
    cv_result_df = run_cross_validation(config_name, cfg, N_FOLDS, CV_SEEDS,
                                         train_df, val_df, test_df, train_dir, val_dir, test_dir)
    cv_rows.append(cv_result_df)

cv_results_df = pd.concat(cv_rows, ignore_index=True) if cv_rows else pd.DataFrame()
cv_results_df


In [ ]:
# =========================================================
# CV SUMMARY (mean +/- std across folds) -> CSV
# =========================================================
def summarize_cv(cv_results_df, csv_path="results/cv_summary.csv"):
    summary = (
        cv_results_df
        .groupby("base_config")[["acc", "kappa"]]
        .agg(["mean", "std", "min", "max", "count"])
    )
    summary.columns = ["_".join(c) for c in summary.columns]
    summary = summary.reset_index()
    summary.to_csv(csv_path, index=False)
    print(f"Saved CV summary -> {csv_path}")
    return summary


if len(cv_results_df):
    cv_summary_df = summarize_cv(cv_results_df)
    cv_summary_df


## Was the original single-split result representative?

Your originally reported number (Stage 5, single train/valid/test split) was
**acc = 0.9044, kappa = 0.9518**. That came from one particular way of splitting the data --
5-fold CV tells you whether that split happened to be a lucky one, an unlucky one, or a
typical one, by showing where it falls relative to the spread across 5 different splits.

In [ ]:
# =========================================================
# ORIGINAL SINGLE-SPLIT RESULT vs 5-FOLD CV SPREAD
# =========================================================
ORIGINAL_RUN = dict(acc=0.9043715846994536, kappa=0.9518186808037806)

def compare_original_to_cv(cv_results_df, original_run=ORIGINAL_RUN, config_name="full_model"):
    fold_rows = cv_results_df[cv_results_df["base_config"] == config_name]
    if len(fold_rows) == 0:
        print(f"No CV rows found for config={config_name} -- run the CV sweep first.")
        return None

    for metric in ["acc", "kappa"]:
        vals = fold_rows[metric].values
        mean, std = vals.mean(), vals.std(ddof=1)
        lo, hi = mean - 1.96 * std / np.sqrt(len(vals)), mean + 1.96 * std / np.sqrt(len(vals))
        orig = original_run[metric]

        z = (orig - mean) / std if std > 0 else float("nan")
        within_1sd = abs(orig - mean) <= std

        print(f"\n{metric.upper()}")
        print(f"  5-fold CV:      mean={mean:.4f}  std={std:.4f}  range=[{vals.min():.4f}, {vals.max():.4f}]")
        print(f"  95% CI of mean: ({lo:.4f}, {hi:.4f})")
        print(f"  Original run:   {orig:.4f}  ({z:+.2f} SD from CV mean, {'within' if within_1sd else 'OUTSIDE'} 1 SD)")

    comparison_df = pd.DataFrame({
        "fold": list(fold_rows["fold"]) + ["original_split"],
        "acc": list(fold_rows["acc"]) + [original_run["acc"]],
        "kappa": list(fold_rows["kappa"]) + [original_run["kappa"]],
    })
    comparison_df.to_csv("results/original_vs_cv_comparison.csv", index=False)
    return comparison_df


# Run once cv_results_df exists (i.e. after the CV sweep cell above has executed):
comparison_df = compare_original_to_cv(cv_results_df)


## Export everything to Excel

Pulls the four result tables produced above into one `.xlsx` workbook, one sheet per source, so you
have a single file to attach to a report. Each CSV is also still
saved on its own (unchanged) -- this just adds a consolidated copy on top.

In [ ]:
# =========================================================
# EXPORT ALL RESULTS TO ONE EXCEL WORKBOOK
# =========================================================
def export_all_results_to_excel(xlsx_path="results/all_results.xlsx"):
    csv_files = {
        "ablation_results":   "results/ablation_results.csv",
        "cv_results":         "results/cv_results.csv",
        "cv_summary":         "results/cv_summary.csv",
        "original_vs_cv":     "results/original_vs_cv_comparison.csv",
    }

    sheets = {}
    for sheet_name, path in csv_files.items():
        if os.path.exists(path):
            sheets[sheet_name] = pd.read_csv(path)
        else:
            print(f"Skipping '{sheet_name}': {path} not found yet")

    if not sheets:
        print("Nothing to export yet -- run the ablation/CV cells above first.")
        return None

    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        for sheet_name, df in sheets.items():
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)  # Excel caps sheet names at 31 chars

    print(f"Saved {len(sheets)} sheet(s) -> {xlsx_path}")
    return sheets


exported_sheets = export_all_results_to_excel()
